<img src="../assets/ga-logo.png" style="float: left; margin: 20px; height: 55px">

# DBSCAN Practice - Solution

---

You're now familiar with how DBSCAN works. Let's practice it in `scikit-learn`.

We will start out working with the [NHL data](https://github.com/josephnelson93/GA-DSI/blob/master/NHL_Data_GA.csv). We're going to investigate clustering teams based on their counting stats.

In [ ]:
%pip install -qqq numpy pandas scikit-learn matplotlib


In [ ]:
import pandas as pd
from sklearn.cluster import DBSCAN
from sklearn.preprocessing import StandardScaler
from sklearn import metrics

import matplotlib.pyplot as plt
%matplotlib inline
%config InlineBackend.figure_format = 'retina'

### 1.  Load our data and perform any basic cleaning and/or EDA.


In [ ]:
nhl = pd.read_csv('./data/nhl.csv')
nhl.head()

In [ ]:
# first, check out dtypes
print(nhl.dtypes) # TOI is an object - let's parse it

In [ ]:
# grab the first four digits, make them an integer
nhl.TOI = nhl.TOI.apply(lambda x: x.split(':')[0])

In [ ]:
# now make it an int (or to_numeric)
nhl.TOI = nhl.TOI.apply(lambda x: int(x))

In [ ]:
print(nhl.dtypes)

### 2. Set up an `X` matrix to perform clustering with DBSCAN.

Let's cluster on all features EXCEPT team and rank.

Make rank be our `y` vector which we can use to do cluster validation. 

In [ ]:
X = nhl.drop(['Team', 'Rank', 'PTS'], axis=1)
print(X.head(2))
y = nhl.Rank
print(y[0:5])

### 3. Scatter plot EDA

Make two scatter plots. At least one axis in one of the plots should represent points (goals for, GA). Do we obtain a general idea from the scatter plots of how many clusters we should expect to extract with a clustering algorithm?

In [ ]:
# remember columns we can plot
print(X.columns)

In [ ]:
# Goals against vs. goals for
plt.scatter(X['GA'], X['GF'])

In [ ]:
# Goals for percent vs. time on ice
plt.scatter(X['GF%'], X['TOI'])

In [ ]:
# honestly looks like there aren't really many viable clusters.

## 4. Scale our data

Standardize the data and compare at least one of the scatterplots for the scaled data to unscaled above.

In [ ]:
Xs = StandardScaler().fit_transform(X)
Xs = pd.DataFrame(Xs, columns=X.columns)

In [ ]:
# Goals against vs. goals for
plt.scatter(Xs['GA'], Xs['GF'])

In [ ]:
# Goals for percent vs. time on ice
plt.scatter(Xs['GF%'], Xs['TOI'])

### 5. Fit a DBSCAN clusterer

Remember to pass an `eps` and `min_samples` of your choice.

In [ ]:
from sklearn.cluster import DBSCAN

dbscn = DBSCAN(eps = 3, min_samples = 3)
dbscn.fit(Xs)

# 'eps' is the max distance between two samples in order for them to be considered in the some cluster.
# min_samples = Minimum number of samples required for a cluster to be considered a cluster.

### 6. Check out the assigned cluster labels

Using the `.labels_` command on our DBSCAN class

In [ ]:
labels = dbscn.labels_  
print(labels) # comprehension: what do these mean? How many are there?
# Observations that did not make it into a DB qualified cluster will recieve the label of -1

In [ ]:
# how many clusters do we have?
n_clusters_ = len(set(labels)) - (1 if -1 in labels else 0)
print(n_clusters_)

### 7. Evaluate the DBSCAN clusters

**7.1 Check the silhouette score.**

How are the clusters?

If you're feeling adventurous, see how you can adjust our epsilon and min_points to improve this.

In [ ]:
print(("Silhouette Coefficient: %0.3f"
      % metrics.silhouette_score(X, labels)))
# Clusters are terrible.

**7.2 Check the homogeneity, completeness, and V-measure against the stored rank `y`**

In [ ]:
print(('Estimated number of clusters: %d' % n_clusters_))
print(("Homogeneity: %0.3f" % metrics.homogeneity_score(y, labels)))
print(("Completeness: %0.3f" % metrics.completeness_score(y, labels)))
print(("V-measure: %0.3f" % metrics.v_measure_score(y, labels)))

In [ ]:
import numpy as np
core_samples = np.zeros_like(labels, dtype = bool)  
core_samples[dbscn.core_sample_indices_] = True 
print(core_samples)

### 8. Plot the clusters

You can choose any two variables for the axes.

In [ ]:
unique_labels = np.unique(labels)
colors = plt.cm.Spectral(np.linspace(0,1, len(unique_labels)))

for (label, color) in zip(unique_labels, colors):
    class_member_mask = (labels == label)
    n = Xs.loc[class_member_mask & core_samples, :]
    plt.plot(n.iloc[:,0],n.iloc[:,1], 'o', markerfacecolor = color, markersize = 10)
    
    n = Xs.loc[class_member_mask & ~core_samples, :]
    plt.plot(n.iloc[:,0],n.iloc[:,1], 'o', markerfacecolor = color, markersize = 5)

In [ ]:
# DBSCAN may be a poor choice for data that is densely populated in one area,
# yet distant from the rest of our data. Frustratingly, we can't find a good fit 
# because of this! (We may have seen this coming in our scatter plots -- even 
# after the data was scaled.) Perhaps, however, learning we have a number of 
# data points are outside our typical clusters is just the insight we needed.

# In these cases, we have observed a moment where the almighty DBSCAN doesn't
# provide tremendous insight.

# For a picture of what is going on behind the scenes, use our handy DBSCAN 
# visualization tool: www.naftaliharris.com/blog/visualizing-dbscan-clustering/, 
# and select the "Packed Circles" example.

### 9. Fit DBSCAN on an easier dataset

Import the `make_circles` function from `sklearn.datasets`. You can use this to create some fake clusters that will perform well with DBSCAN.

Create some `X` and `y` using the function. Here is some sample code:
```python
from sklearn.datasets import make_circles
circles_X, circles_y = make_circles(n_samples=1000, random_state=123, noise=0.1, factor=0.2)
```

**9.1 Plot the fake circles data.**

In [ ]:
from sklearn.datasets import make_circles
circles_X, circles_y = make_circles(n_samples=1000, random_state=123, noise=0.1, factor=0.2)

plt.scatter(circles_X[:,0], circles_X[:,1])

**9.2 Scale the data and fit DBSCAN on it.**

In [ ]:
X = StandardScaler().fit_transform(circles_X)

In [ ]:
X.shape

In [ ]:
plt.scatter(X[:,0], X[:,1]) #how can we cluster this?!
# This is just me thinking out loud, but an RBF kernel for an SVM would be spot on.

In [ ]:
dbscn = DBSCAN(eps = .5, min_samples = 3).fit(X)

In [ ]:
labels = dbscn.labels_  
print(labels) # comprehension: what do these mean? How many are there?

In [ ]:
# how many clusters do we have?
n_clusters_ = len(set(labels)) - (1 if -1 in labels else 0)
print(n_clusters_)

**9.3 Evaluate DBSCAN visually, with silhouette, and with the metrics against the true `y`.**

In [ ]:
print(("Silhouette Coefficient: %0.3f"
      % metrics.silhouette_score(X, labels)))

In [ ]:
print(('Estimated number of clusters: %d' % n_clusters_))
print(("Homogeneity: %0.3f" % metrics.homogeneity_score(circles_y, labels)))
print(("Completeness: %0.3f" % metrics.completeness_score(circles_y, labels)))
print(("V-measure: %0.3f" % metrics.v_measure_score(circles_y, labels)))

In [ ]:
import numpy as np
core_samples = np.zeros_like(labels, dtype = bool)  
core_samples[dbscn.core_sample_indices_] = True 
print(core_samples)

In [ ]:
unique_labels = np.unique(labels)
colors = plt.cm.Spectral(np.linspace(0,1, len(unique_labels)))

for (label, color) in zip(unique_labels, colors):
    class_member_mask = (labels == label)
    n = X[class_member_mask & core_samples]
    plt.plot(n[:,0],n[:,1], 'o', markerfacecolor = color, markersize = 10)
    
    n = X[class_member_mask & ~core_samples]
    plt.plot(n[:,0],n[:,1], 'o', markerfacecolor = color, markersize = 5)

In [ ]:
# DBSCAN performs perfectly in this case! When we have anisotropicly or circularly 
# plotted data, we should opt for DBSCAN because KMEANS has a number of 
# assumptions (http://scikit-learn.org/stable/auto_examples/cluster/plot_kmeans_assumptions.html) 
# that often don't pan out.